# Análisis Exploratorio de Calidad del Aire
## Posadas, Misiones, Argentina

Este notebook realiza un análisis exploratorio detallado de los datos de calidad del aire en Posadas.

**Contaminantes analizados:**
- PM2.5 - Material particulado fino
- PM10 - Material particulado
- NO₂ - Dióxido de nitrógeno
- O₃ - Ozono troposférico
- CO - Monóxido de carbono
- SO₂ - Dióxido de azufre

In [ ]:
# Importar librerías
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Módulos del proyecto
from src.data_processing import AirQualityProcessor
from src.visualization import AirQualityVisualizer
from config.config import LOCATION, POLLUTANTS

# Configurar visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Librerías cargadas correctamente")

## 1. Cargar Datos

In [ ]:
# Cargar datos procesados
processor = AirQualityProcessor()

# Si tienes datos procesados, cárgalos aquí
# df = pd.read_csv('../data/processed/processed_data.csv')

# Para este ejemplo, usaremos datos sintéticos
df_raw = pd.read_csv('../data/raw/synthetic_data.csv')
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])

print(f"📊 Datos cargados: {len(df_raw)} registros")
print(f"📅 Período: {df_raw['datetime'].min()} a {df_raw['datetime'].max()}")
print(f"🏷️ Contaminantes: {df_raw['parameter'].unique()}")

df_raw.head()

## 2. Procesamiento de Datos

In [ ]:
# Limpiar datos
df = processor.clean_data(df_raw.copy())

# Agregar características temporales
df = processor.add_temporal_features(df)

# Calcular AQI
df = processor.add_aqi_column(df)

print("✓ Datos procesados")
df.info()

## 3. Estadísticas Descriptivas

In [ ]:
# Calcular estadísticas
stats = processor.calculate_statistics(df)

# Mostrar tabla
print("="*80)
print("ESTADÍSTICAS DESCRIPTIVAS POR CONTAMINANTE")
print("="*80)
print(stats.to_string(index=False))

# Guardar tabla
stats.to_csv('../output/reports/estadisticas_descriptivas.csv', index=False)
print("\n💾 Estadísticas guardadas en output/reports/estadisticas_descriptivas.csv")

## 4. Análisis de Distribuciones

In [ ]:
# Crear visualizador
viz = AirQualityVisualizer()

# Gráfico de distribuciones
viz.plot_distributions(df, save=True)
plt.show()

## 5. Series Temporales

In [ ]:
# Gráfico de series temporales
viz.plot_time_series(df, save=True)
plt.show()

## 6. Patrones Temporales

In [ ]:
# Mapa de calor por hora y día de la semana (PM2.5)
viz.plot_heatmap_by_time(df, pollutant='pm25', save=True)
plt.show()

# Repetir para NO2 (contaminante relacionado con tráfico)
viz.plot_heatmap_by_time(df, pollutant='no2', save=True, filename='heatmap_no2.png')
plt.show()

## 7. Variación Estacional

In [ ]:
# Patrones estacionales
viz.plot_seasonal_patterns(df, save=True)
plt.show()

## 8. Índice de Calidad del Aire (AQI)

In [ ]:
# Distribución de categorías AQI
viz.plot_aqi_distribution(df, save=True)
plt.show()

# Resumen de AQI
print("\nRESUMEN DE ÍNDICE DE CALIDAD DEL AIRE")
print("="*60)
print(f"AQI Promedio: {df['aqi'].mean():.1f}")
print(f"AQI Mediana: {df['aqi'].median():.1f}")
print(f"AQI Máximo: {df['aqi'].max():.0f}")
print(f"\nDistribución por categoría:")
print(df['aqi_category'].value_counts())

## 9. Correlaciones entre Contaminantes

In [ ]:
# Matriz de correlación
viz.plot_correlation_matrix(df, save=True)
plt.show()

## 10. Excedencias de Guías OMS

In [ ]:
# Calcular excedencias
exceedances = []

for param in ['pm25', 'pm10', 'no2', 'so2']:
    param_data = df[df['parameter'] == param]
    
    if not param_data.empty and 'who_guideline_24h' in POLLUTANTS[param]:
        guideline = POLLUTANTS[param]['who_guideline_24h']
        
        # Calcular promedios diarios
        daily_avg = param_data.groupby(param_data['datetime'].dt.date)['value'].mean()
        
        total_days = len(daily_avg)
        exceeded_days = (daily_avg > guideline).sum()
        percentage = (exceeded_days / total_days) * 100
        
        exceedances.append({
            'Contaminante': POLLUTANTS[param]['name'],
            'Guía OMS (24h)': f"{guideline} {POLLUTANTS[param]['unit']}",
            'Días totales': total_days,
            'Días excedidos': exceeded_days,
            'Porcentaje': f"{percentage:.1f}%"
        })

exceedances_df = pd.DataFrame(exceedances)
print("\nEXCEDENCIAS DE GUÍAS OMS (Promedios de 24 horas)")
print("="*80)
print(exceedances_df.to_string(index=False))

# Guardar
exceedances_df.to_csv('../output/reports/excedencias_oms.csv', index=False)

## 11. Conclusiones Preliminares

Basado en el análisis exploratorio:

1. **Calidad del Aire General**: [Analizar los resultados del AQI]
2. **Contaminantes Críticos**: [Identificar cuáles exceden las guías OMS]
3. **Patrones Temporales**: [Describir patrones horarios/semanales/estacionales]
4. **Correlaciones**: [Interpretar relaciones entre contaminantes]
5. **Recomendaciones**: [Sugerir análisis adicionales o acciones]

In [ ]:
# Guardar datos procesados para análisis posteriores
df.to_csv('../data/processed/processed_data.csv', index=False)
print("\n💾 Datos procesados guardados en data/processed/processed_data.csv")
print("✓ Análisis exploratorio completado")